# 23 · Pipeline and expert parallelism; sizing a 70B model
Level L4 · Part 3 · core · ~25 min · builds on 19, 22

## Goal
By the end of this lesson you can pick a tensor-parallel degree and a replica count for the 70B model at a target request rate. You can state the memory and communication cost of each choice, and say when pipeline and expert parallelism are the right tools.

## Why it matters
- **Without it:** you copy an 8-GPU command from a blog, or you add cards until the start-up error stops. Nobody knows how many replicas the support desk needs at peak, so the queue grows in the morning.
- **With it:** a few lines turn the model size, the GPU type and a target request rate into a GPU count. They also show what each split costs per token and in all-reduces.
- **When you don't need it:** a model that fits on one GPU with room for its cache needs no split; add replicas instead. Pipeline parallelism matters when a group is bigger than one node. Expert parallelism matters only for mixture-of-experts models.

## The idea
A **group** is the set of GPUs that serve one copy of the model. **Tensor parallelism** (lesson 22) cuts every layer across the GPUs of one group, so all of them work on each token and must join their results after every layer. **Pipeline parallelism** cuts the model into **stages**, one stage being one pipeline slice: a run of consecutive layers on its own tensor-parallel group. A token passes through the stages in order. **Expert parallelism** puts the experts of a mixture-of-experts (MoE) model on different GPUs, so a token visits only the few experts it is routed to.

A **replica** is one full copy of the group behind a load balancer. Replicas add requests per second; splits add memory and cost communication.

Think of a kitchen. Tensor parallel is eight cooks on one dish, talking after each step. Pipeline is an assembly line: each station adds one part and passes the dish on.

## Picture
One token through a model cut into two stages, each stage a tensor-parallel group of four GPUs:
```mermaid
flowchart TB
  M["one token enters the model"] --> S1["stage 1: layers 1 to 40 on GPUs 1 to 4 (TP=4)"]
  S1 -->|"send one 16 KiB activation"| S2["stage 2: layers 41 to 80 on GPUs 5 to 8 (TP=4)"]
  S2 --> O["one token out"]
```

## Step by step
### Step 1: the numbers
These are copies of public numbers, as in lessons 19 and 22: the Llama 3.1 70B `config.json` values, and the L4's 24 GB, 300 GB/s of memory bandwidth and 121 TFLOP/s of dense FP16 maths (NVIDIA's L4 datasheet, as used in lesson 06). FP8 maths runs at 242 TFLOP/s dense on the L4 (NVIDIA's L4 datasheet: 485 with sparsity). **FP8** stores each weight in 1 byte and **4-bit** in half a byte (FP16 uses 2); lesson 20 shows how those formats are made. The fixed cost of one all-reduce is an assumption, as in lesson 22. Nothing is downloaded.

In [1]:
import math

GIB, GB = 2**30, 1e9
L4_USABLE = 24e9 * 0.92                       # one L4 for vLLM (lesson 19)
BW = 300e9                                    # L4 memory bandwidth, bytes per second
FLOPS = {"FP16": 121e12, "FP8": 242e12}       # L4 dense tensor maths (NVIDIA L4 datasheet)
PCIE = 32e9                                   # PCIe Gen4 x16, one direction (lesson 22)
FIXED_US = 20                                 # ASSUMPTION: fixed cost of one all-reduce call

PARAMS_70B = 70_553_706_496
LAYERS, Q_HEADS, KV_HEADS = 80, 64, 8
KV_PER_TOKEN = 2 * LAYERS * KV_HEADS * 128 * 2     # FP16 cache, all GPUs together (320 KiB)
ACT_PER_TOKEN = 8192 * 2                           # one token's activation vector (16 KiB)
ALLREDUCES = 2 * LAYERS                            # attention and MLP, one each per layer: 160 per pass
PROMPT, ANSWER = 1_600, 200                        # a Brightside request (lesson 19)
CTX = PROMPT + ANSWER                              # cache tokens per request

print(f"70B cache: {KV_PER_TOKEN // 1024} KiB per token; all-reduces per pass: {ALLREDUCES}")

70B cache: 320 KiB per token; all-reduces per pass: 160


**What you should see:** 320 KiB of cache per token and 160 all-reduces per pass. Both numbers come from the model, not from the number of GPUs.

### Step 2: the cost of one all-reduce and one stage hop
A ring all-reduce moves `2 × (N − 1) / N` of the message through each GPU. A stage boundary is simpler: one point-to-point send of the activations. The helpers below price both over PCIe.

In [2]:
def allreduce_s(tokens, tp, link=PCIE):
    """One ring all-reduce of the activations of `tokens` tokens over tp GPUs, in seconds."""
    if tp == 1:
        return 0.0
    moved = 2 * (tp - 1) / tp * tokens * ACT_PER_TOKEN
    return moved / link + FIXED_US * 1e-6

def hop_s(tokens, link=PCIE):
    """One send of `tokens` tokens' activations across a pipeline stage boundary."""
    return tokens * ACT_PER_TOKEN / link + FIXED_US * 1e-6

def pass_s(tokens, tp):
    """All-reduce time of one forward pass of `tokens` tokens (160 calls)."""
    return ALLREDUCES * allreduce_s(tokens, tp)

print(f"decode, 1 token:   TP=8 all-reduces {pass_s(1, 8) * 1e3:.1f} ms, one stage hop {hop_s(1) * 1e6:.0f} us")
print(f"prefill, 1,600 tokens: TP=8 all-reduces {pass_s(PROMPT, 8) * 1e3:.0f} ms, one stage hop {hop_s(PROMPT) * 1e3:.2f} ms")

decode, 1 token:   TP=8 all-reduces 3.3 ms, one stage hop 21 us
prefill, 1,600 tokens: TP=8 all-reduces 233 ms, one stage hop 0.84 ms


**What you should see:** a decode pass costs about 3 ms of all-reduces on TP=8, but a 1,600-token prefill costs about 230 ms. One stage hop costs about 20 µs for one token and under a millisecond for the whole prompt. Lesson 22 priced the same all-reduce for the prefill.

### Step 3: tensor or pipeline, for one token
In a tensor-parallel group, all GPUs read their shard of the weights at the same time, so the time per token is the weights divided by `tp × bandwidth`. In a pipeline the stages run one after another, so for one request their times add up. `token_ms` gives both costs for one token.

In [3]:
def token_ms(tp, stages, bpp=2):
    """One token through the whole model: weight reads (ms) and communication (ms)."""
    read = stages * (PARAMS_70B * bpp / (stages * tp)) / BW   # stages run in turn
    comm = pass_s(1, tp) + (stages - 1) * hop_s(1)
    return read * 1e3, comm * 1e3

print("FP16 70B, one request:")
for tp, stages in ((8, 1), (4, 2), (2, 4), (1, 8)):
    read, comm = token_ms(tp, stages)
    w = PARAMS_70B * 2 / (tp * stages) / GIB
    print(f"  TP={tp} PP={stages} ({tp * stages} GPUs): {w:4.1f} GiB per GPU, "
          f"read {read:5.1f} ms + comm {comm:4.1f} ms = {read + comm:5.1f} ms per token")

FP16 70B, one request:
  TP=8 PP=1 (8 GPUs): 16.4 GiB per GPU, read  58.8 ms + comm  3.3 ms =  62.1 ms per token
  TP=4 PP=2 (8 GPUs): 16.4 GiB per GPU, read 117.6 ms + comm  3.3 ms = 120.9 ms per token
  TP=2 PP=4 (8 GPUs): 16.4 GiB per GPU, read 235.2 ms + comm  3.3 ms = 238.5 ms per token
  TP=1 PP=8 (8 GPUs): 16.4 GiB per GPU, read 470.4 ms + comm  0.1 ms = 470.5 ms per token


**What you should see:** TP=8 gives the fastest token. Pipeline stages do not speed up one request: TP=4 with PP=2 takes about twice as long to read the weights. Comm stays at 3.3 ms in the TP=4 PP=2 row, because each all-reduce is dominated by its fixed cost. What pipeline changes is where the traffic goes: all-reduces stay inside a stage's smaller group, and a stage boundary needs only one 16 KiB send. That is why vLLM's parallelism docs put the tensor split inside a node and the pipeline split across nodes (lesson 34).

### Step 4: mixture of experts, where expert parallelism helps
The 70B model is dense: every weight is used for every token, so there are no experts to spread. Mixtral 8x7B is a mixture-of-experts model: each layer has 8 experts and a router picks 2 for each token. Its **total** parameters set the memory, and its **active** parameters set the bytes read per token. The numbers are from Mistral's announcement.

In [4]:
MIX_TOTAL, MIX_ACTIVE = 46.7e9, 12.9e9       # Mixtral 8x7B, total and active per token
EXPERTS, TOP_K = 8, 2

print(f"Mixtral FP16 weights in memory: {MIX_TOTAL * 2 / GIB:.1f} GiB (every expert)")
print(f"bytes read per token: Mixtral {MIX_ACTIVE * 2 / GB:.1f} GB, dense 70B {PARAMS_70B * 2 / GB:.1f} GB "
      f"({PARAMS_70B / MIX_ACTIVE:.1f} times more)")
print(f"expert parallel over {EXPERTS} GPUs: one expert of {EXPERTS} per GPU, "
      f"about {MIX_TOTAL * 2 / EXPERTS / GIB:.1f} GiB each (rough: the router and attention weights come on top)")
print(f"experts touched per token per layer: {TOP_K} of {EXPERTS}")

Mixtral FP16 weights in memory: 87.0 GiB (every expert)
bytes read per token: Mixtral 25.8 GB, dense 70B 141.1 GB (5.5 times more)
expert parallel over 8 GPUs: one expert of 8 per GPU, about 10.9 GiB each (rough: the router and attention weights come on top)
experts touched per token per layer: 2 of 8


**What you should see:** Mixtral needs 87 GiB of memory, but reads about 5.5 times fewer bytes per token than the dense 70B. Expert parallelism spreads the memory: one expert of eight per GPU. Each token then goes to the GPUs that hold its two experts. In vLLM this is the `--enable-expert-parallel` flag. It changes only the expert layers: they are split by expert across the same GPUs instead of by tensor slice (vLLM data-parallel page). The dense 70B model has no experts, so there is nothing for the flag to split.

### Step 5: size the Brightside group
The target is the desk's peak: `TARGET_RPS` requests per second, each with a 1,600-token prompt and a 200-token answer. For each FP16, FP8 and 4-bit option:
- the group size is the TP degree, and the cache holds `kv_tokens // 1,800` requests (lesson 19);
- one request costs the group its prefill time plus its share of the decode steps, where the batch shares each step;
- replicas = target × headroom ÷ requests per second per replica.

Every line is a best case: peak maths, no overlap of communication and compute, a full cache. Two numbers are assumptions and are labelled.

In [5]:
TARGET_RPS = 6       # the desk's peak requests per second (your number)
HEADROOM = 1.3       # ASSUMPTION: 30% spare for bursts and one failed replica
OPTIONS = [          # name, bytes per parameter, TP degree, maths format
    ("FP16", 2, 8, "FP16"),
    ("FP8", 1, 4, "FP8"),
    ("4-bit", 0.5, 2, "FP16"),   # 4-bit weights, FP16 maths (assumed)
]

`fit` checks that a split divides the heads and leaves room for the cache. `step_s` prices one decode step for a batch of requests.

In [6]:
def fit(bpp, tp):
    """Weights per GPU, room left per GPU, and cache tokens for one group of tp GPUs."""
    if Q_HEADS % tp or KV_HEADS % tp:
        raise ValueError(f"TP={tp} must divide {Q_HEADS} query and {KV_HEADS} KV heads")
    w = PARAMS_70B * bpp / tp
    room = L4_USABLE - w
    if room <= 0:
        raise ValueError(f"TP={tp} at {bpp} bytes per parameter does not fit")
    return w, room, room * tp / KV_PER_TOKEN

def step_s(bpp, tp, batch):
    """One decode step for `batch` requests: weight reads, cache reads and all-reduces.
    One step serves the whole batch, so its cost is shared by every request in it."""
    read = (PARAMS_70B * bpp / tp + batch * CTX * KV_PER_TOKEN / tp) / BW
    return read + ALLREDUCES * allreduce_s(batch, tp)

`plan_option` puts the pieces together for one option: the cache, the batch, the cost of one request and the replicas.

In [7]:
def plan_option(bpp, tp, maths):
    _, _, kv_tokens = fit(bpp, tp)
    batch = int(kv_tokens // CTX)                           # requests whose cache fits
    prefill_maths = 2 * PARAMS_70B * PROMPT / (tp * FLOPS[maths])
    prefill_comm = pass_s(PROMPT, tp)                       # the all-reduces of the same pass
    prefill = prefill_maths + prefill_comm
    decode = ANSWER * step_s(bpp, tp, batch) / batch        # this request's share of the steps
    rps = 1 / (prefill + decode)                            # requests one group finishes per second
    replicas = math.ceil(TARGET_RPS * HEADROOM / rps)
    return {"kv_tokens": kv_tokens, "batch": batch, "prefill": prefill, "decode": decode,
            "prefill_maths": prefill_maths, "prefill_comm": prefill_comm,
            "rps": rps, "replicas": replicas, "total": replicas * tp}

Now plan each option and print what it costs.

In [8]:
results = {}
for name, bpp, tp, maths in OPTIONS:
    r = plan_option(bpp, tp, maths)
    results[name] = r
    print(f"{name:6} TP={tp}: cache {r['kv_tokens']:>7,.0f} tokens, batch {r['batch']:>2}, "
          f"prefill {r['prefill']:.2f} s (maths {r['prefill_maths']:.2f} + all-reduces {r['prefill_comm']:.2f}) "
          f"+ decode {r['decode']:.2f} s per request")
    print(f"       {r['rps']:.2f} requests/s per replica -> {r['replicas']} replicas "
          f"= {r['total']} GPUs for {TARGET_RPS} requests/s at peak")

best = min(results, key=lambda k: results[k]["total"])
print(f"fewest GPUs: {best}")

FP16   TP=8: cache 108,437 tokens, batch 60, prefill 0.47 s (maths 0.23 + all-reduces 0.23) + decode 0.28 s per request
       1.33 requests/s per replica -> 6 replicas = 48 GPUs for 6 requests/s at peak
FP8    TP=4: cache  54,218 tokens, batch 30, prefill 0.43 s (maths 0.23 + all-reduces 0.20) + decode 0.54 s per request
       1.03 requests/s per replica -> 8 replicas = 32 GPUs for 6 requests/s at peak
4-bit  TP=2: cache  27,109 tokens, batch 15, prefill 1.07 s (maths 0.93 + all-reduces 0.13) + decode 1.04 s per request
       0.47 requests/s per replica -> 17 replicas = 34 GPUs for 6 requests/s at peak
fewest GPUs: FP8


**What you should see:** the table gives the GPUs per replica, the cache, the batch and the two costs per request. At TP=8 in FP16, the prefill takes about twice what its maths alone says (0.47 s against 0.23 s), because the all-reduces over a 1,600-token prompt cost as much as the maths. At TP=4 in FP8 the all-reduces add about 0.2 s to 0.23 s of maths. FP8 on TP=4 needs the fewest GPUs (32). The 4-bit option is close on GPUs (34), but it needs 17 replicas: prefill maths on 2 GPUs takes 0.93 s and the batch of 15 is half of FP8's 30. These are planning floors; lesson 27 measures them.

### Step 6: the commands (walkthrough, not run here)
Run these on a GPU host, after checking each flag with `vllm serve --help=all` on the version you install. This lesson does not start a server.

**1. The chosen group: FP8, one node, four GPUs.**
```bash
vllm serve <your FP8 70B checkpoint> --tensor-parallel-size 4 --gpu-memory-utilization 0.92
```
You should see four workers in the log, and `/health` answers once all four are ready.

**2. A group bigger than one node: TP inside, PP across.**
```bash
vllm serve <model path> --tensor-parallel-size 4 --pipeline-parallel-size 2
```
Two nodes need the multi-node setup of lesson 34. Each node runs its own 4-GPU tensor group.

**3. A mixture-of-experts model.**
```bash
vllm serve <MoE model path> --tensor-parallel-size 8 --enable-expert-parallel
```
The flag changes how the expert layers are split: by expert, across the same GPUs as the tensor group. Read the parallelism page for your version before you size it, and check the flag names with `--help=all` first.

## At work
- **vLLM's parallelism guide** sets tensor parallelism to the GPUs per node and pipeline parallelism to the node count; without NVLink, it uses pipeline instead of tensor parallelism. → lesson 34.
- **DeepSeek's public notes** (Feb 2025) ran expert parallelism over many nodes: prefill on 4 (EP32), decode on 18 (EP144). → lesson 34.
- **vLLM's data-parallel page** combines data-parallel attention with expert or tensor-parallel MoE layers; its example is DP=4, TP=2 on one 8-GPU node. → lesson 34.

Go deeper: expert load balancing and the pipeline-bubble formula.

Sources: vLLM docs (docs.vllm.ai, opened 2026-10-21) · Mistral AI, "Mixtral of experts" · DeepSeek, open-infra-index day 6.

## Common mistakes
**1. Choosing a TP degree that does not divide the heads.** Six GPUs look like one group, but 64 query heads do not split into six equal parts. The split refuses this.

In [9]:
fit(1, 6)   # FP8 weights on six GPUs in one group: TP=6, and 64 heads cannot be split in six

ValueError: TP=6 must divide 64 query and 8 KV heads

The fix is two levels: TP=2 inside each group and PP=3 across three stages, using all six GPUs (`TP=2 × PP=3`). This check covers memory only: FP8 70B weights are about 70 GB, so about 11.8 GB per GPU over six GPUs, which fits in the 22 GB usable on an L4. PP=3 splits the 80 layers unevenly, which vLLM supports.

**2. Sizing replicas from the decode speed alone.** The decode share is only part of a request's cost. At TP=8 in FP16 on PCIe, prefill all-reduces cost as much as prefill maths. Count both, then add headroom.

## Try it
Your team has four L4s and must run the FP8 70B model. Use `token_ms` to compare TP=4 with TP=2 × PP=2 for one request: the time per token and the memory per GPU. Which one is faster for one request?

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
for tp, stages in ((4, 1), (2, 2)):
    read, comm = token_ms(tp, stages, bpp=1)
    w = PARAMS_70B * 1 / (tp * stages) / GIB
    print(f"TP={tp} PP={stages}: {w:.1f} GiB per GPU, {read + comm:.1f} ms per token")
print("TP=4 is faster for one request: the stages of the pipeline run one after another.")

TP=4 PP=1: 16.4 GiB per GPU, 62.1 ms per token
TP=2 PP=2: 16.4 GiB per GPU, 120.9 ms per token
TP=4 is faster for one request: the stages of the pipeline run one after another.


## Recap
- Tensor parallelism shards every layer: 160 all-reduces per pass.
- Pipeline stages do not speed up one request. They cut all-reduce traffic to smaller groups, and a boundary sends one 16 KiB activation.
- Expert parallelism spreads an MoE model's experts; the dense 70B model has none.
- Size the group from memory, then count replicas from group-seconds.

**Interview:** "How would you serve a 70B model at 6 requests per second on L4s?" Strong answer: FP16 needs 141 GB, so TP=8, or FP8 on TP=4. Keep TP inside a node: on PCIe, its all-reduces cost as much as the maths. Count replicas from group-seconds with headroom; call them upper bounds until a load test confirms them.